# Phase 3 walkthrough — ML surrogates for DC-OPF LMPs

Reads the outputs of `python -m src.run --phases 3`. Sections: (1) the naive baseline and why it is exact without
congestion, (2) accuracy by bus and regime, (3) the main result — the Monte Carlo study rerun with the surrogate,
(4) the step-function failure analysis, (5) SHAP, (6) a live prediction with the trained models.

In [ ]:
import os, sys, json, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from IPython.display import Image, Markdown, display
from src.config import load_config, data_dir
cfg = load_config()
d = data_dir(cfg)
if not (d / "phase3" / "models.pkl").exists():
    from src.run import main; main(["--phases", "3"])
T = lambda name, **kw: pd.read_csv(f"reports/tables/{name}.csv", **kw)
summary = json.load(open("reports/phase3_summary.json"))

## 1. Baseline: the merit-order price
With no binding line, the DC-OPF LMP at every bus equals the copper-plate price: the offer of the cheapest unit
still needed after stacking all cheaper available capacity. It is computed analytically (no LP) and is exact on all
17,520 hourly cases. The surrogates receive it as a feature and learn only LMP − merit price (the congestion part).

In [ ]:
from src.features import merit_order_price
from src.run import load_result
x = pd.read_parquet(d / "phase1" / "inputs.parquet")
copper = load_result(d / "phase1", "copper_")
p = merit_order_price(cfg, x)
print("max |merit-order price − copper-plate LP price| over 17,520 h:", float((p - copper.lmp['CAL']).abs().max()))

## 2. Accuracy on the 20,000 held-out MC scenarios

In [ ]:
pd.DataFrame(summary["mc_task"]["summary_by_model"]).set_index("model").round(2)

In [ ]:
display(Image("reports/figures/surrogate_mae_by_regime.png", width=760))
T("surrogate_mc_regimes", index_col=0).round(2)

In [ ]:
T("surrogate_ablation").set_index(["model", "target", "loss"]).round(2)

## 3. Main result: the Monte Carlo study with the surrogate

In [ ]:
display(Image("reports/figures/surrogate_mc_congestion.png", width=740))
T("surrogate_mc_congestion", index_col=0).round(4)

In [ ]:
display(Image("reports/figures/surrogate_lmp_ecdf.png", width=860))
T("surrogate_mc_quantiles").set_index(["bus", "stat"]).round(2)

In [ ]:
display(Image("reports/figures/surrogate_runtime.png", width=740))
display(T("surrogate_runtime", index_col=0).round(4))
print(f"break-even: {summary['mc_task']['break_even_scenarios']:,.0f} scenarios")
display(Image("reports/figures/surrogate_learning_curve.png", width=640))

## 4. LMP is a step function — where each model fails
One test hour, one input varied over 101 points, the OPF re-solved at each point.

In [ ]:
display(Image("reports/figures/surrogate_step_wind.png", width=900))
display(Image("reports/figures/surrogate_step_load.png", width=900))
pd.DataFrame(summary["sweep_mae"]).T.round(2)

## 5. TreeSHAP

In [ ]:
display(Image("reports/figures/surrogate_shap_importance.png", width=900))
display(Image("reports/figures/surrogate_shap_dependence.png", width=700))

## 6. Live prediction with the trained models
Take a test scenario, push South wind to 95 %, and compare the surrogates with a fresh OPF solve.

In [ ]:
from src.montecarlo import Scenarios
from src.features import build_features
from src.simulate import solve_block
art = pickle.load(open(d / "phase3" / "models.pkl", "rb"))
scen = Scenarios(*(pd.read_parquet(d / "phase2" / f"mc_{k}.parquet") for k in ("inputs", "line_out", "gen_avail")))
i = scen.inputs.index[scen.line_out.sum(axis=1) == 0][7]
rows = scen.inputs.loc[[i, i]].copy(); rows.index = pd.RangeIndex(2, name="case")
rows.loc[1, "cf_wind_south"] = 0.95
ga = scen.gen_avail.loc[[i, i]].set_axis(rows.index)
lo = scen.line_out.loc[[i, i]].set_axis(rows.index)
X = build_features(cfg, rows, lo, ga)[art["feature_columns"]]
truth = solve_block(cfg, rows, gen_availability=ga).lmp.add_prefix("lmp_")
out = {"OPF": truth}
out.update({k: m.predict(X) for k, m in art["models"].items()})
res = pd.concat(out, names=["model", "case"]).round(2)
res.index = res.index.set_levels(["as sampled", "South wind 95%"], level="case")
display(res)
art["classifier"].predict_proba(X).round(3).set_axis(["as sampled", "South wind 95%"])